<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# ROS 2 Workspaces and Packages

The previous notebooks used a command-line publisher to investigate messages, discovery and names. That publisher supplied ready-made behaviour: send a specified message at a specified rate on a specific topic.

A custom application needs its own source code. How is that code organised, which libraries does it require, and how does ROS 2 know which function to run?

This notebook looks at the package supplied with this learning experience, `ros2_basics`: a publisher, a subscriber and a node that drives the Duckiebot, built in a standard **ament_python** package. Inspecting it connects the source files to the information needed to build and run them. By the end of this notebook, the files responsible for application behaviour, dependencies, executable registration, launching and configuration will be identifiable.

## Know the exercise workspace

A **workspace** is a directory in which packages are built and installed together. The editor records its workspace location in the `COLCON_WS` environment variable. Open a terminal and run:

```bash
printf 'Workspace: %s\n' "$COLCON_WS"
cd /code/src/lx-ros2-basics
pwd
```

which shows the workspace and the location of this learning experience inside the editor:

```shell
Workspace: /code
/code/src/lx-ros2-basics
```

The two paths serve different purposes. `/code` is the workspace root, where built packages are installed (`/code/build`, `/code/install`, `/code/log` will appear after the first build). `/code/src/lx-ros2-basics` is this repository, mounted into the editor: it contains the notebooks and the `packages/` directory with the source code. The following commands use paths relative to the repository root.

A typical ROS 2 workspace contains four directories, each serving a different role in the software development process:

- **source space** (`src/`): the source code, organised in packages. This is where you edit;
- **build space** (`build/`): intermediate files produced while building;
- **install space** (`install/`): the built packages, ready to be used. This is where executables, launch files and configuration are read from when you run something, together with a `setup.bash` script that makes the install space visible to a terminal;
- **log space** (`log/`): the logs of every build, useful to understand a build error.

The ROS 2 distribution itself is a workspace installed at `/opt/ros/jazzy`, and `duckietown_msgs` lives in another one at `/opt/duckietown/ros2/duckietown_msgs`. Workspaces are layered: these two are the **underlay**, and `/code` is your **overlay** on top of them.

## Find the supplied package

**colcon** is the tool used to discover and build packages. Its `list` command inspects source packages without building them. From the repository root:

```bash
colcon list --base-paths packages
```

The supplied package appears as:

```shell
ros2_basics	packages/ros2_basics	(ros.ament_python)
```

The three entries identify the package name, its source directory and its package type. `ament_python` identifies the Python packaging approach used by this ROS 2 package: pure Python, installed with `setuptools`, no compilation. (ROS 2 packages that contain C++ code, or that define messages, use the `ament_cmake` build type instead.)

Finding the package establishes that `colcon` recognises its source, but it does not mean the package was built, that its executables are available to `ros2 run`, or that it is configured correctly.

## What makes up an ament_python package?

A package brings together application code and the information needed to install and use it. The files surrounding the Python code reveal those different responsibilities:

```bash
find packages/ros2_basics -type f | sort
```

<figure style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    packages/ros2_basics/
    |-- package.xml                    the manifest: name, dependencies, build type
    |-- setup.py                       installation: executables, data files
    |-- setup.cfg                      where the executables are installed
    |-- resource/
    |   `-- ros2_basics                marker for the ament index (empty file)
    |-- ros2_basics/                   the Python package (importable)
    |   |-- __init__.py
    |   |-- my_publisher_node.py
    |   |-- my_subscriber_node.py
    |   `-- drive_node.py
    |-- launch/
    |   |-- my_publisher_node.launch.py
    |   |-- my_subscriber_node.launch.py
    |   `-- drive_node.launch.py       how to start each node (namespace, parameters)
    `-- config/
        |-- my_publisher_node/default.yaml
        |-- my_subscriber_node/default.yaml
        `-- drive_node/default.yaml    default parameter values of each node
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em;">
    Figure 1: Directory structure of the <code>ros2_basics</code> package.
  </figcaption>
</figure>

The repeated directory name reflects two layers: the outer `ros2_basics` directory holds the ROS 2 package, while the inner `ros2_basics` directory is the importable Python package that implements the nodes. Before running any of them, the package must describe what it needs and how its code should be installed.

### The manifest: `package.xml`

The package manifest declares the identity of the package and its ROS dependencies. Open [package.xml](../packages/ros2_basics/package.xml) in the editor, or print it:

```bash
cat packages/ros2_basics/package.xml
```

Among the package metadata, the following lines identify its name, its build type and its dependencies:

```xml
<name>ros2_basics</name>

<buildtool_depend>ament_python</buildtool_depend>

<exec_depend>rclpy</exec_depend>
<exec_depend>std_msgs</exec_depend>
<exec_depend>std_srvs</exec_depend>
<exec_depend>sensor_msgs</exec_depend>
<exec_depend>duckietown_msgs</exec_depend>
<exec_depend>lx_ros2_tools</exec_depend>

<export>
  <build_type>ament_python</build_type>
</export>
```

`rclpy` provides the Python client library; `std_msgs`, `sensor_msgs` and `duckietown_msgs` supply the message definitions used by the nodes (`String`, `CompressedImage`, `Twist2DStamped`, `WheelEncoderStamped`); `std_srvs` the `Trigger` service type; `lx_ros2_tools` the small helpers of this learning experience (live parameters, image conversion). Declaring a dependency records a requirement; it does not itself install the required software.

You can replace `YOUR_FULL_NAME` with your first and last name and `YOUR_EMAIL@EXAMPLE.COM` with your email address, in `package.xml` and in `setup.py`.

### How does an executable name reach the Python code? `setup.py`

A source filename alone does not register a ROS 2 executable. The installation configuration provides that connection. Open [setup.py](../packages/ros2_basics/setup.py). Near the end of the file, the `console_scripts` entries associate executable names with Python functions:

```python
entry_points={
    "console_scripts": [
        "my_publisher_node = ros2_basics.my_publisher_node:main",
        "my_subscriber_node = ros2_basics.my_subscriber_node:main",
        "drive_node = ros2_basics.drive_node:main",
    ],
},
```

The first entry can be read from left to right:

| Part | Meaning |
| --- | --- |
| `my_publisher_node` | executable name to register during installation (what `ros2 run` uses) |
| `ros2_basics.my_publisher_node` | Python module containing the implementation |
| `main` | function called when that executable starts |

An executable name identifies something that can be launched. A node name identifies a participant in the running graph. Registering an executable does not, by itself, determine the node name used by its implementation (here they happen to coincide).

The `data_files` list of `setup.py` installs everything that is not Python code but must still be found at run time: the manifest, the ament-index marker, the launch files and the configuration files:

```python
data_files=[
    ("share/ament_index/resource_index/packages", ["resource/" + package_name]),
    ("share/" + package_name, ["package.xml"]),
    (os.path.join("share", package_name, "launch"), glob("launch/*.launch.py")),
    (os.path.join("share", package_name, "config", "drive_node"), glob("config/drive_node/*.yaml")),
    ...
],
```

A launch file or a yaml file that is not listed here is simply not installed, and `ros2 launch` will not find it.

### Where will the executables be installed? `setup.cfg`

The function mapping explains what should run, but ROS 2 also needs to find the installed executable:

```bash
cat packages/ros2_basics/setup.cfg
```

```ini
[develop]
script_dir=$base/lib/ros2_basics
[install]
install_scripts=$base/lib/ros2_basics
```

These settings direct executable installation into the package-specific `lib/ros2_basics` directory beneath the installation base, where `ros2 run` expects to find the package's executables. `$base` is an installation placeholder.

The remaining `resource/ros2_basics` marker is installed by `setup.py` into the ament package index (notebook 01), which is how `ros2 pkg` learns about the installed package.

### Launch files and configuration

Two directories go beyond the minimal package of the ROS 2 tutorials. Open [launch/drive_node.launch.py](../packages/ros2_basics/launch/drive_node.launch.py). A **launch file** is a Python script that describes how to start one or more nodes. This one declares a `veh` argument (the robot name, defaulting to the `VEHICLE_NAME` environment variable) and starts one node in that namespace, with the parameters of a yaml file:

```python
Node(
    package="ros2_basics",
    executable="drive_node",
    name="drive_node",
    namespace=veh,
    parameters=[config],
    ...
)
```

The yaml file [config/drive_node/default.yaml](../packages/ros2_basics/config/drive_node/default.yaml) holds the default values of the node's **parameters**:

```yaml
/**:
  ros__parameters:
    v: 0.2
    omega: 0.0
    rate_hz: 10.0
    max_duration: 5.0
    enabled: false
```

`/**:` means "for any node this file is given to". Keeping defaults in a file rather than in the Python code lets you tune a node without editing (or rebuilding) it. Notebook 09 walks through both files in detail.

Table 2 summarises how these files connect the source code to an installed ROS 2 package.

| File or directory | Responsibility |
| --- | --- |
| `ros2_basics/*.py` | implement the application behaviour |
| `package.xml` | declare package metadata, ROS dependencies and build type |
| `setup.py` | configure the Python installation, executable entry points and data files |
| `setup.cfg` | select the executable installation directories |
| `resource/ros2_basics` | provide the package-index marker |
| `launch/*.launch.py` | describe how to start the nodes (namespace, parameters, remappings) |
| `config/<node>/default.yaml` | default parameter values of each node |

*Table 2: Responsibilities of the package files.*

## Identify the right file before making a change

Each file has its own purpose, so edits should be made in the appropriate places. Suppose the publisher should send different text by default: that change belongs in its yaml file, or in its Python implementation. If the package needs an additional ROS dependency, the manifest should describe it. If an executable should call a different function, its entry-point registration in `setup.py` must change. If a node should start in another namespace, its launch file changes.

These are different kinds of change, even though they belong to the same package. Identifying the responsible file helps avoid changing installation settings when the intended change concerns application behaviour.

The supplied package already contains the structure needed for this learning experience. Creating a new package from scratch is a separate workflow (`ros2 pkg create --build-type ament_python <name>`), described in the [ROS 2 package-creation tutorial](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Creating-Your-First-ROS2-Package.html).

The source inspection located `ros2_basics`, identified its dependencies, and traced executable names to Python functions. Building and sourcing the package makes its installed executables available to ROS 2: that is the next notebook.

## Further reading

The [ROS 2 Jazzy package tutorial](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Creating-Your-First-ROS2-Package.html) covers creating a package, its required files and the steps from source to an installed executable. The [colcon documentation](https://colcon.readthedocs.io/en/released/user/quick-start.html) describes workspaces.

Let's move to [the next notebook, where we build the package](../notebooks/06_build_and_discover_a_package.ipynb).